In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from econml.dml import LinearDML
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

train = pd.read_csv("../Data/Processed/train.csv")
test = pd.read_csv("../Data/Processed/test.csv")
print(train.shape, test.shape)

(51200, 13) (12800, 13)


In [2]:
TREATMENT_COL = "segment"
OUTCOME_COL = "conversion"
CONTROL = "No E-Mail"
ID_COL = "Customer ID"

FEATURE_COLS = ["recency", "history", "mens", "womens", "newbie", "history_floor", "zip_code", "channel"]

LGBM_PARAMS = dict(n_estimators=200, learning_rate=0.05, num_leaves=15, min_child_samples=100,
                   subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)

cat_ohe = ["zip_code", "channel"]

In [3]:
preprocessor = ColumnTransformer(
    transformers=[
        ("passthrough_cols", "passthrough", ["recency", "mens", "womens", "newbie", "history_floor"]),
        ("history_log", FunctionTransformer(np.log1p, feature_names_out="one-to-one"), ["history"]),
        ("cat_ohe", OneHotEncoder(drop=["Rural", "Multichannel"], handle_unknown="ignore", sparse_output=False), cat_ohe),
    ],
    remainder="drop", verbose_feature_names_out=False,
).set_output(transform="pandas")

In [4]:
class DMLPipeline:
    
    def __init__(self, preprocessor, estimator, feature_cols, treatment_col, outcome_col, control):
        self.preprocessor = preprocessor
        self.estimator = estimator
        self.feature_cols = feature_cols
        self.treatment_col = treatment_col
        self.outcome_col = outcome_col
        self.control = control
        
    def fit(self, df):
        X = self.preprocessor.fit_transform(df[self.feature_cols])    
        self.estimator.fit(Y=df[self.outcome_col].values, T=df[self.treatment_col].values, X=X.values)
        self.arms_ = [a for a in sorted(df[self.treatment_col].unique()) if a != self.control]
        self.feature_names_ = list(X.columns)
        return self
    
    def effects(self, df):
        X = self.preprocessor.transform(df[self.feature_cols]).values
        out = {f"ite_{arm.replace(' E-Mail', '').lower()}":
               np.ravel(self.estimator.effect(X, T0=self.control, T1=arm)) for arm in self.arms_}
        return pd.DataFrame(out, index=df.index)
        
    def recommend(self, df):
        eff = self.effects(df)
        best = pd.Series(self.arms_, index=eff.columns).reindex(eff.idxmax(axis=1)).values
        best = np.where(eff.max(axis=1) > 0, best, self.control)
        return eff.assign(best_segment=best)

In [5]:
dml=LinearDML(
    model_t=LGBMClassifier(**LGBM_PARAMS),
    model_y=LGBMClassifier(**LGBM_PARAMS),
    discrete_outcome=True,
    discrete_treatment=True,
    cv=5,
    random_state=42
)

dml_pipe=DMLPipeline(preprocessor,dml,FEATURE_COLS,TREATMENT_COL,OUTCOME_COL,CONTROL)

dml_pipe.fit(train)

print("fitted. e-mail arms compared with", CONTROL, "->", dml_pipe.arms_)
print("features seen by the model:", dml_pipe.feature_names_)

fitted. e-mail arms compared with No E-Mail -> ['Mens E-Mail', 'Womens E-Mail']
features seen by the model: ['recency', 'mens', 'womens', 'newbie', 'history_floor', 'history', 'zip_code_Suburban', 'zip_code_Urban', 'channel_Phone', 'channel_Web']


In [6]:
rec = dml_pipe.recommend(test)
rec.insert(0, ID_COL, test[ID_COL].values)
display(rec.head())

print("average estimated effect on the test set (percentage points):")
print((rec[["ite_mens", "ite_womens"]].mean() * 100).round(3).to_string())
print("\nspread across customers (std, pp):")
print((rec[["ite_mens", "ite_womens"]].std() * 100).round(3).to_string())
print("\nbest option per customer:")
print(rec["best_segment"].value_counts().to_string())

,Customer ID,ite_mens,ite_womens,best_segment
0,C-43252,0.007159,0.000417,Mens E-Mail
1,C-7410,0.004933,0.007095,Womens E-Mail
2,C-6444,0.008296,0.005739,Mens E-Mail
3,C-29292,0.005400,-0.000919,Mens E-Mail
4,C-9606,0.008020,0.003897,Mens E-Mail


average estimated effect on the test set (percentage points):
ite_mens      0.677
ite_womens    0.316

spread across customers (std, pp):
ite_mens      0.341
ite_womens    0.363

best option per customer:
best_segment
Mens E-Mail      10319
Womens E-Mail     2240
No E-Mail          241


In [7]:
def uplift_by_bucket(test_df, ite, arm, n_buckets=5):
    d = test_df[[TREATMENT_COL, OUTCOME_COL]].copy()
    d["ite"] = np.asarray(ite)
    d = d[d[TREATMENT_COL].isin([CONTROL, arm])].copy()
    d["bucket"] = pd.qcut(d["ite"], n_buckets, labels=[f"B{i}" for i in range(1, n_buckets + 1)])  
    rows = []
    for b, g in d.groupby("bucket", observed=True):
        treated, control = g[g[TREATMENT_COL] == arm][OUTCOME_COL], g[g[TREATMENT_COL] == CONTROL][OUTCOME_COL]
        rows.append({"bucket": b, "customers": len(g), "predicted uplift (pp)": g["ite"].mean() * 100,
                     "treated conv. (%)": treated.mean() * 100, "control conv. (%)": control.mean() * 100,
                     "observed uplift (pp)": (treated.mean() - control.mean()) * 100})
    return pd.DataFrame(rows).round(3)

for arm, col in [("Mens E-Mail", "ite_mens"), ("Womens E-Mail", "ite_womens")]:
    print(f"\n{arm} vs {CONTROL}")
    print(uplift_by_bucket(test, rec[col], arm).to_string(index=False))


Mens E-Mail vs No E-Mail
bucket  customers  predicted uplift (pp)  treated conv. (%)  control conv. (%)  observed uplift (pp)
    B1       1705                  0.189              1.575              0.490                 1.085
    B2       1704                  0.568              0.700              0.472                 0.228
    B3       1704                  0.699              0.713              0.928                -0.215
    B4       1704                  0.813              1.218              0.340                 0.878
    B5       1705                  1.135              1.995              0.586                 1.409

Womens E-Mail vs No E-Mail
bucket  customers  predicted uplift (pp)  treated conv. (%)  control conv. (%)  observed uplift (pp)
    B1       1708                 -0.195              0.336              0.982                -0.646
    B2       1708                  0.120              1.453              0.567                 0.886
    B3       1707                  0.

In [8]:
p_convert_pipe=Pipeline([
    ('preprocess', clone(preprocessor)),
    ('model',LGBMClassifier(**LGBM_PARAMS))
])

p_convert_pipe.fit(train[FEATURE_COLS],train[OUTCOME_COL])

rec["p_convert"]=p_convert_pipe.predict_proba(test[FEATURE_COLS])[:,1]
print("test AUC of p_convert:", round(roc_auc_score(test[OUTCOME_COL], rec["p_convert"]), 3), "(modest signal is expected: only 0.9% convert)")

scored_test = rec[[ID_COL, "p_convert", "ite_mens", "ite_womens", "best_segment"]]
scored_test.to_csv("../Data/Processed/scored_test.csv", index=False)
scored_test.head()

test AUC of p_convert: 0.604 (modest signal is expected: only 0.9% convert)


,Customer ID,p_convert,ite_mens,ite_womens,best_segment
0,C-43252,0.011296,0.007159,0.000417,Mens E-Mail
1,C-7410,0.006699,0.004933,0.007095,Womens E-Mail
2,C-6444,0.006844,0.008296,0.005739,Mens E-Mail
3,C-29292,0.005102,0.005400,-0.000919,Mens E-Mail
4,C-9606,0.016517,0.008020,0.003897,Mens E-Mail


Testing another model with Random Forest

In [9]:
RF_PARAMS = dict(n_estimators=300, min_samples_leaf=100, max_features=0.5,
                 n_jobs=-1, random_state=42)

In [10]:
dml_rf = LinearDML(
    model_t=RandomForestClassifier(**RF_PARAMS),
    model_y=RandomForestClassifier(**RF_PARAMS),
    discrete_outcome=True,
    discrete_treatment=True,
    cv=5,
    random_state=42
)

dml_rf_pipe = DMLPipeline(clone(preprocessor), dml_rf, FEATURE_COLS, TREATMENT_COL, OUTCOME_COL, CONTROL)
dml_rf_pipe.fit(train)

print("fitted. e-mail arms compared with", CONTROL, "->", dml_rf_pipe.arms_)
print("features seen by the model:", dml_rf_pipe.feature_names_)

fitted. e-mail arms compared with No E-Mail -> ['Mens E-Mail', 'Womens E-Mail']
features seen by the model: ['recency', 'mens', 'womens', 'newbie', 'history_floor', 'history', 'zip_code_Suburban', 'zip_code_Urban', 'channel_Phone', 'channel_Web']


In [11]:
rec_rf = dml_rf_pipe.recommend(test)
rec_rf.insert(0, ID_COL, test[ID_COL].values)
display(rec_rf.head())

print("average estimated effect on the test set (percentage points):")
print((rec_rf[["ite_mens", "ite_womens"]].mean() * 100).round(3).to_string())
print("\nspread across customers (std, pp):")
print((rec_rf[["ite_mens", "ite_womens"]].std() * 100).round(3).to_string())
print("\nbest option per customer:")
print(rec_rf["best_segment"].value_counts().to_string())

,Customer ID,ite_mens,ite_womens,best_segment
0,C-43252,0.007249,0.000451,Mens E-Mail
1,C-7410,0.004408,0.006788,Womens E-Mail
2,C-6444,0.007849,0.005532,Mens E-Mail
3,C-29292,0.005295,-0.000796,Mens E-Mail
4,C-9606,0.007924,0.003880,Mens E-Mail


average estimated effect on the test set (percentage points):
ite_mens      0.677
ite_womens    0.313

spread across customers (std, pp):
ite_mens      0.344
ite_womens    0.355

best option per customer:
best_segment
Mens E-Mail      10277
Womens E-Mail     2323
No E-Mail          200


In [12]:
for arm, col in [("Mens E-Mail", "ite_mens"), ("Womens E-Mail", "ite_womens")]:
    print(f"\n{arm} vs {CONTROL}")
    print(uplift_by_bucket(test, rec_rf[col], arm).to_string(index=False))


Mens E-Mail vs No E-Mail
bucket  customers  predicted uplift (pp)  treated conv. (%)  control conv. (%)  observed uplift (pp)
    B1       1705                  0.196              1.697              0.365                 1.331
    B2       1704                  0.559              0.581              0.593                -0.012
    B3       1704                  0.693              0.237              0.813                -0.576
    B4       1704                  0.805              1.707              0.452                 1.255
    B5       1705                  1.149              1.993              0.587                 1.406

Womens E-Mail vs No E-Mail
bucket  customers  predicted uplift (pp)  treated conv. (%)  control conv. (%)  observed uplift (pp)
    B1       1708                 -0.185              0.337              1.102                -0.765
    B2       1708                  0.121              1.327              0.455                 0.872
    B3       1707                  0.

In [13]:
p_convert_pipe_rf = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", RandomForestClassifier(**RF_PARAMS))
])

p_convert_pipe_rf.fit(train[FEATURE_COLS], train[OUTCOME_COL])

rec_rf["p_convert"] = p_convert_pipe_rf.predict_proba(test[FEATURE_COLS])[:, 1]
print("test AUC of p_convert (RF):", round(roc_auc_score(test[OUTCOME_COL], rec_rf["p_convert"]), 3))

scored_test_rf = rec_rf[[ID_COL, "p_convert", "ite_mens", "ite_womens", "best_segment"]]
scored_test_rf.to_csv("../Data/Processed/scored_test_rf.csv", index=False)   # separate file: does not overwrite the LGBM output
scored_test_rf.head()

test AUC of p_convert (RF): 0.616


,Customer ID,p_convert,ite_mens,ite_womens,best_segment
0,C-43252,0.011657,0.007249,0.000451,Mens E-Mail
1,C-7410,0.009607,0.004408,0.006788,Womens E-Mail
2,C-6444,0.007253,0.007849,0.005532,Mens E-Mail
3,C-29292,0.007333,0.005295,-0.000796,Mens E-Mail
4,C-9606,0.019644,0.007924,0.003880,Mens E-Mail


## Comparing LGBM vs Random Forest

In [14]:
# 1) Nuisance-model quality (out-of-fold, on train). Lower log-loss is better.
Xtr = clone(preprocessor).fit_transform(train[FEATURE_COLS])
skf = StratifiedKFold(5, shuffle=True, random_state=42)
y, t = train[OUTCOME_COL].values, train[TREATMENT_COL].values

print(f"outcome baseline (constant prob) log-loss: {log_loss(y, np.full(len(y), y.mean())):.5f}")
for name, mdl in [("LGBM", LGBMClassifier(**LGBM_PARAMS)), ("RF  ", RandomForestClassifier(**RF_PARAMS))]:
    p = cross_val_predict(mdl, Xtr, y, cv=skf, method="predict_proba")[:, 1]
    print(f"{name} outcome model  log-loss {log_loss(y, p):.5f}   AUC {roc_auc_score(y, p):.3f}")

# treatment was randomised, so the ideal treatment model predicts ~1/3 for everyone (log-loss ~ ln 3)
print(f"\ntreatment baseline (ln 3) log-loss: {np.log(3):.5f}")
for name, mdl in [("LGBM", LGBMClassifier(**LGBM_PARAMS)), ("RF  ", RandomForestClassifier(**RF_PARAMS))]:
    p = cross_val_predict(mdl, Xtr, t, cv=skf, method="predict_proba")
    print(f"{name} treatment model log-loss {log_loss(t, p):.5f}")

outcome baseline (constant prob) log-loss: 0.05156
LGBM outcome model  log-loss 0.05293   AUC 0.550
RF   outcome model  log-loss 0.05221   AUC 0.562

treatment baseline (ln 3) log-loss: 1.09861
LGBM treatment model log-loss 1.10173
RF   treatment model log-loss 1.10064


## Conclusion

**Setup.** `LinearDML` with 5-fold cross-fitting estimated the effect of each e-mail arm versus *No E-Mail* on conversion. The models were trained on 51,200 customers and scored on 12,800 held-out customers. LightGBM is the main nuisance learner, and a Random Forest with the same preprocessing, folds and seed is the robustness check.

**Findings**
- **Average effect (test set):** Mens e-mail +0.68 pp, Womens e-mail +0.31 pp. Both learners agree, and the values match the DoWhy estimates (+0.680 / +0.312 pp).
- **Heterogeneity:** The effect varies across customers (std ≈ 0.34 pp for Mens, ≈ 0.36 pp for Womens). For Womens the spread is larger than the average, and the lowest-ranked customers have a negative predicted effect.
- **Recommendations:** About 10.3k of the 12.8k test customers are best served by the Mens e-mail, about 2.2–2.3k by the Womens e-mail, and about 200–240 (under 2%) by no e-mail.
- **LGBM vs Random Forest:** Practically identical. RF is marginally better on nuisance log-loss (outcome 0.0522 vs 0.0529) and on `p_convert` AUC (0.616 vs 0.604), but these gaps are within sampling noise (~115 conversions in the test set).
- **Nuisance models:** Neither outcome model beats the constant-probability baseline on log-loss (0.0516), so conversion is hard to predict from these features (CV AUC ≈ 0.55). The treatment models do no better than 1/3 per arm (ln 3 ≈ 1.099), as expected for randomised assignment.
- **Why the learners agree:** The final stage of `LinearDML` is linear in the features, so the nuisance learner only changes the residualisation, not the shape of the effect.

**Caveats**
- Only ~0.9% of customers convert, so the decile/bucket tables are noisy (standard error ≈ 0.45 pp per bucket) and observed uplift is not monotone in predicted uplift. The average effects are reliable. The individual-level ranking is weak evidence.
- Recommendations use the sign of a point estimate, with no confidence intervals.

**Decision and next steps.** Keep the LightGBM pipeline as the main model, since Random Forest gives no meaningful gain. Next:

 (1) compare ranking quality with Qini curves and policy value,
 
  (2) try a non-linear final stage (e.g. `CausalForestDML`) for richer heterogeneity,
  
   (3) use the ITEs for the budget allocation across e-mail arms.